# Chequeos de PlantDoc — sesión 4, bloque 3 (sin ningún modelo)

Duplicados por phash entre las 13 carpetas candidatas (train + test juntos), par con etiquetas contradictorias, acción propuesta (**no se ejecuta nada**) y material para revisar etiquetas de `Tomato leaf` y `Bell_pepper leaf`.
Sin modelos, sin métricas, nada en `data/splits/`, ninguna carpeta renombrada. Config: `configs/00c_chequeos_plantdoc.yaml`.

In [ ]:
import subprocess, time
import numpy as np, pandas as pd, yaml
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from PIL import Image
from foliares.utils.paths import cargar_rutas
from foliares.data import plantdoc_chequeos as pc
try:
    import psutil; _proc = psutil.Process()
except ImportError:
    _proc = None
def mem_mb(): return round(_proc.memory_info().rss / 2**20) if _proc else None

pd.set_option("display.width", 250, "display.max_rows", 200, "display.max_columns", 40, "display.max_colwidth", 80)
R = cargar_rutas()
CFG = yaml.safe_load((R.repo / "configs" / "00c_chequeos_plantdoc.yaml").read_text(encoding="utf-8"))
U6, U10 = CFG["duplicados"]["umbral_estricto"], CFG["duplicados"]["umbral_exploratorio"]
h = subprocess.check_output(["git", "-C", str(R.plantdoc), "log", "-1", "--format=%H"], text=True).strip()
print("PlantDoc", h, "OK" if h == CFG["commit_plantdoc"] else "NO COINCIDE")
FIG = R.repo / CFG["salida_figuras"]; FIG.mkdir(parents=True, exist_ok=True)
T0 = time.time()
meta = pc.listar_plantdoc(R, R.repo / CFG["renombrados_csv"])
print(len(meta), "imágenes en las", meta.clase.nunique(), "carpetas candidatas")
display(meta.groupby(["clase", "split"]).size().unstack(fill_value=0))

## 3a. Duplicados por phash (64 bits, 8 variantes: 4 rotaciones × espejado)

Todos los pares (i<j) sobre train+test juntos, de cualquier clase. La distancia es el mínimo entre la identidad de una imagen y las 8 variantes de la otra (en ambos sentidos). ≤6 estricto; 7–10 exploratorio.

In [ ]:
t = time.time()
h8, info = pc.hashes_con_cache(R, meta, R.interim / CFG["duplicados"]["cache_hashes"])
D, var = pc.distancias(h8)
P = pc.tabla_pares(meta, D, var, U10)
print("hashing:", info, f"| distancias+pares {time.time()-t:.1f} s | RSS {mem_mb()} MB")
tab = P.assign(rango=np.where(P.distancia <= U6, "<=6 (estricto)", "7-10 (exploratorio)")).groupby(["categoria", "rango"]).size().unstack(fill_value=0)
tab["<=10 (acumulado)"] = tab.sum(axis=1)
display(tab.reindex(pc.CATEGORIAS).fillna(0).astype(int))
print("Imágenes de test involucradas en pares <=6:", len(set(P[(P.distancia <= U6)].pipe(lambda x: list(x.i[x.split_a == "test"]) + list(x.j[x.split_b == "test"])))))

In [ ]:
S = P[P.distancia <= U6]
grupos = {c: S[S.categoria == c].reset_index(drop=True) for c in pc.CATEGORIAS}
print("test↔test <=10:", len(P[P.categoria == pc.CATEGORIAS[2]]), "pares (no hay ninguno: no hay nada que mirar)")
pc.grilla_pares(grupos[pc.CATEGORIAS[0]], R, FIG / "pares_train_test.png", "misma clase train↔test (<=6)")
cd = grupos[pc.CATEGORIAS[3]]
pc.grilla_pares(cd.iloc[:16], R, FIG / "pares_clases_distintas_1.png", f"clases distintas (<=6) 1/2 — {len(cd)} pares en total")
pc.grilla_pares(cd.iloc[16:], R, FIG / "pares_clases_distintas_2.png", "clases distintas (<=6) 2/2")
tt = P[(P.categoria == pc.CATEGORIAS[1])].reset_index(drop=True)
pc.grilla_pares(tt.iloc[:15], R, FIG / "pares_train_train_1.png", f"misma clase train↔train (<=10) 1/2 — {len(tt)} pares en total")
pc.grilla_pares(tt.iloc[15:], R, FIG / "pares_train_train_2.png", "misma clase train↔train (<=10) 2/2")
for f in sorted(FIG.glob("pares_*.png")): print(f.relative_to(R.repo).as_posix())

In [ ]:
# Listado completo con ambas rutas y distancia (nombres originales; ruta relativa a DATA_ROOT en el CSV)
for c in pc.CATEGORIAS:
    x = P[P.categoria == c]
    print(f"\n### {c}: {len(x)} pares")
    display(x[["distancia", "split_a", "clase_a", "nombre_original_a", "split_b", "clase_b", "nombre_original_b"]].reset_index(drop=True))

## 3b. Par con etiquetas contradictorias (`irish-blight-symptoms…`)

Lado a lado, con nombres completos. **No se decide la etiqueta**; solo se reporta la evidencia.

In [ ]:
import hashlib
x = P[P.nombre_original_a.str.startswith("irish-blight-symptoms") | P.nombre_original_b.str.startswith("irish-blight-symptoms")]
assert len(x) == 1; f = x.iloc[0]
pa, pb = R.absoluta(f.ruta_a), R.absoluta(f.ruta_b)
fig, axs = plt.subplots(1, 2, figsize=(11, 5))
for ax, p, s, c, n in ((axs[0], pa, f.split_a, f.clase_a, f.nombre_original_a), (axs[1], pb, f.split_b, f.clase_b, f.nombre_original_b)):
    with Image.open(p) as im: ax.imshow(im.convert("RGB"))
    ax.set_title(f"{s}/{c}\n{n}", fontsize=8); ax.axis("off")
fig.savefig(FIG / "par_irish_blight.png", dpi=80); plt.close(fig)
def ficha(p):
    b = p.read_bytes()
    with Image.open(p) as im: dims = im.size
    return {"tamaño_bytes": len(b), "md5": hashlib.md5(b).hexdigest(), "ancho_x_alto": dims}
fa, fb = ficha(pa), ficha(pb)
display(pd.DataFrame([{"ruta": f.ruta_a, **fa}, {"ruta": f.ruta_b, **fb}]))
print("Distancia phash:", f.distancia, "| bytes idénticos:", fa["md5"] == fb["md5"])
print("Mismo nombre de archivo en ambas carpetas:", f.nombre_original_a == f.nombre_original_b)
print("Nombre completo:", f.nombre_original_a)
# Otras copias del mismo nombre o de la misma foto en las 28 carpetas de PlantDoc (train y test)
base = f.nombre_original_a.rsplit(".", 1)[0]
otros = [p.relative_to(R.plantdoc).as_posix() for p in R.plantdoc.rglob("*") if p.is_file() and base[:30] in p.name]
print("Archivos de PlantDoc cuyo nombre contiene el prefijo del archivo:", otros)
print("Pista en la ruta: carpeta A =", f.clase_a, "(train); carpeta B =", f.clase_b, "(test). El nombre del archivo dice 'irish-blight' —"
      " nombre común del tizón tardío de la papa (Phytophthora infestans)— y no menciona 'early'/'Alternaria'. Es una pista, no una etiqueta.")
print("Figura:", (FIG / "par_irish_blight.png").relative_to(R.repo).as_posix())

## 3c. Tabla de duplicados con acción propuesta (NO se ejecuta)

Regla propuesta: **sacar de dev, nunca de test**. Por grupo de duplicados (componente de pares ≤6): si hay una copia en test se sacan todas las de train; si no, se conserva una y se sacan las demás; si el grupo mezcla clases (etiquetas contradictorias) no se elige etiqueta: se sacan todas las copias de train. Las imágenes sacadas son *propuestas*; nada se mueve ni se borra.

In [ ]:
P6 = P[P.distancia <= U6]
sacar_A = pc.sacar_de_dev(meta, P6)                                  # solo duplicados de la misma clase
sacar_B = pc.sacar_de_dev(meta, P6, incluir_clases_distintas=True)   # + clases distintas (etiquetas contradictorias)
sacar_C = pc.sacar_de_dev(meta, P, incluir_clases_distintas=True)    # + exploratorio (<=10): cota superior
def coincide(fila, pares):
    na, nb = str(fila.nombre_original_a), str(fila.nombre_original_b)
    return any((na.startswith(x) and nb.startswith(y)) or (na.startswith(y) and nb.startswith(x)) for x, y in pares)
def lectura(fila):
    if fila.umbral != "estricto (<=6)": return "no revisado uno por uno (rango exploratorio)"
    if coincide(fila, CFG["lectura_visual_no_duplicado"]): return "no parece la misma foto (falso positivo probable)"
    if coincide(fila, CFG["lectura_visual_dudosos"]): return "dudoso: misma lesión con otro recorte, o dos fotos"
    return "misma foto (lectura visual en la grilla; a confirmar por el equipo)"
T = P.copy()
T["lectura_visual"] = T.apply(lectura, axis=1)
T["accion_propuesta"] = T.apply(lambda r: pc.accion_propuesta(r, sacar_B), axis=1)
T.insert(0, "id_par", [f"D{k:03d}" for k in range(1, len(T) + 1)])
cols = ["id_par", "categoria", "umbral", "distancia", "variante", "split_a", "clase_a", "ruta_a", "nombre_original_a",
        "split_b", "clase_b", "ruta_b", "nombre_original_b", "lectura_visual", "accion_propuesta"]
T[cols].to_csv(R.repo / CFG["salida_csv"], index=False, encoding="utf-8-sig", lineterminator="\n")
print("Guardado:", CFG["salida_csv"], "|", len(T), "pares")
display(T.groupby(["categoria", "lectura_visual"]).size())
print("Imágenes de dev (train) que la regla propondría sacar: A (misma clase <=6) =", len(sacar_A), "| B (+clases distintas <=6) =", len(sacar_B), "| C (+exploratorio) =", len(sacar_C))

### Cuántas imágenes por clase quedarían en dev y test (solo números)

dev = carpeta `train` oficial; test = carpeta `test` oficial (la partición dev/test propia **no está decidida**). El test nunca pierde imágenes bajo la regla.

In [ ]:
esc = {"A: misma clase <=6": sacar_A, "B: A + clases distintas <=6": sacar_B, "C: B + exploratorio <=10": sacar_C}
base = pc.conteos_dev_test(meta, set())[["clase", "dev_antes", "test"]].set_index("clase")
tabla = base.copy()
for k, s in esc.items():
    tabla[f"dev {k[0]}"] = pc.conteos_dev_test(meta, s).set_index("clase").dev_despues
tabla.loc["TOTAL"] = tabla.sum()
display(tabla)

In [ ]:
# Regla fijada de antemano: se usa el split oficial depurado si cada clase conserva >=5 imágenes de test y dev >=40.
# Se EVALÚA la regla (no se aplica ni se decide): con las 13 carpetas y sin la de arañas (0 imágenes de test).
RL = CFG["regla_split_oficial"]; sin = CFG["clases_sin_test_para_la_regla"]
filas = []
for k, s in esc.items():
    t = pc.conteos_dev_test(meta, s).set_index("clase")
    for nombre, cl in (("13 carpetas", t.index), ("12 (sin " + sin[0] + ")", [c for c in t.index if c not in sin])):
        x = t.loc[cl]
        fallan_test = x.index[x.test < RL["min_test_por_clase"]].tolist(); fallan_dev = x.index[x.dev_despues < RL["min_dev_por_clase"]].tolist()
        filas.append({"escenario": k, "clases": nombre, "cumple": not fallan_test and not fallan_dev,
                      "test<5": fallan_test, "dev<40": fallan_dev, "dev_mínimo": int(x.dev_despues.min()), "test_mínimo": int(x.test.min())})
display(pd.DataFrame(filas).set_index(["escenario", "clases"]))

In [ ]:
# Efecto sobre el test: imágenes de test que tienen una copia en train (la regla NO las saca del test)
tr_te = P6[P6.categoria.isin([pc.CATEGORIAS[0], pc.CATEGORIAS[3]]) & ((P6.split_a == "test") | (P6.split_b == "test"))]
idx_test = sorted(set(list(tr_te.i[tr_te.split_a == "test"]) + list(tr_te.j[tr_te.split_b == "test"])))
tipo = {}
for r in tr_te.itertuples():
    k = r.i if r.split_a == "test" else r.j
    tipo[k] = "misma clase" if (r.categoria == pc.CATEGORIAS[0] and tipo.get(k) != "etiqueta contradictoria") else "etiqueta contradictoria"
et = meta.iloc[idx_test].assign(tipo=[tipo[k] for k in idx_test])
display(et.groupby(["clase", "tipo"]).size().unstack(fill_value=0).assign(test_total=meta[meta.split == "test"].groupby("clase").size()).fillna(0).astype(int))
print("Imágenes de test (de", int((meta.split == "test").sum()), ") con alguna copia en train a distancia <=6:", len(idx_test))

In [ ]:
# Manifiesto de PlantDoc con columnas para AMBAS opciones (split oficial depurado / partición propia agrupada por hash).
comp = pc.componentes(len(meta), list(zip(P6.i, P6.j)))
tam = pd.Series(comp).map(pd.Series(comp).value_counts()).to_numpy()
mp = meta[["ruta", "clase", "split", "nombre_original"]].rename(columns={"split": "split_oficial"}).copy()
mp["grupo_hash"] = [f"H{c:04d}" if t > 1 else "" for c, t in zip(comp, tam)]
mp["tam_grupo_hash"] = tam
mp["clases_en_grupo"] = [meta.clase[comp == c].nunique() for c in comp]
mp["etiquetas_contradictorias"] = mp.clases_en_grupo > 1
mp["propuesta_sacar_de_dev_A"] = [k in sacar_A for k in range(len(meta))]
mp["propuesta_sacar_de_dev_B"] = [k in sacar_B for k in range(len(meta))]
mp["revisar_etiqueta"] = mp.clase.isin(CFG["revision_etiquetas"]["clases"].values())
mp.to_csv(R.interim / CFG["manifiesto_plantdoc"], index=False, lineterminator="\n")
print("Guardado:", R.relativa(R.interim / CFG["manifiesto_plantdoc"]), "| grupos de duplicados (>=2 imágenes):", mp[mp.tam_grupo_hash > 1].grupo_hash.nunique(),
      "| imágenes en grupos:", int((mp.tam_grupo_hash > 1).sum()))
display(mp[mp.tam_grupo_hash > 1].groupby("etiquetas_contradictorias").agg(grupos=("grupo_hash", "nunique"), imagenes=("ruta", "size")))

## 3d. Material para revisar etiquetas de `Tomato leaf` y `Bell_pepper leaf`

Hojas de contacto con **id anónimo** (orden aleatorio con semilla fija: el id no revela split ni orden de archivo) y planilla CSV `id, split, ruta, revisor_1, revisor_2, motivo`. Dos integrantes revisan **por separado** (cada uno trabaja en su copia de la planilla y después se juntan). Criterios (los aplican personas):
1. no es la especie; 2. no es una hoja (fruto, diagrama, lámina con texto); 3. el nombre del archivo nombra una enfermedad **Y** se ven lesiones. **Nunca** por borrosa, recortada o difícil. Se descarta solo si ambos revisores coinciden.

In [ ]:
RV = CFG["revision_etiquetas"]
carpeta = R.interim / RV["carpeta_hojas"]; carpeta.mkdir(parents=True, exist_ok=True)
plan = pc.material_revision(R, meta, RV["clases"], carpeta, RV["semilla"], RV["imagenes_por_hoja"])
estado_planilla = pc.guardar_planilla(plan.sort_values("id"), R.repo / RV["planilla"])   # nunca pisa la planilla: los revisores escriben ahí
clase_de_id = plan.id.str.split("-").str[0].map(RV["clases"])                          # la planilla no tiene columna `clase`
print("Planilla:", RV["planilla"], f"({estado_planilla})", "|", len(plan), "imágenes:", plan.groupby(clase_de_id).size().to_dict(), "| por split:", plan.groupby([clase_de_id, plan.split]).size().to_dict())
print("Hojas de contacto (no versionadas, en", R.relativa(carpeta) + "):")
for p in sorted(carpeta.glob("*.png")): print("  ", p.name, f"{p.stat().st_size/1024:.0f} KB")
display(plan.sort_values("id").head(6))

In [ ]:
# Datos para el registro de accesos al test (docs/bitacora/accesos_test.md)
S6 = P[P.distancia <= U6]
mostradas = set(S6.i[S6.split_a == "test"]) | set(S6.j[S6.split_b == "test"])
rev_test = plan[plan.split == "test"]
print("Imágenes de test mostradas en grillas de pares <=6:", len(mostradas))
print("Imágenes de test en las hojas de contacto de revisión:", len(rev_test), "| distintas de las anteriores:",
      len(set(rev_test.ruta) - set(meta.ruta[sorted(mostradas)])))
print("Imágenes de test hasheadas (13 carpetas):", int((meta.split == "test").sum()))
print(f"Tiempo total del notebook: {time.time()-T0:.0f} s | RSS: {mem_mb()} MB")